# Actividad 3: Aplicación de algoritmos de aprendizaje supervisado con PySpark

**Materia:** Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Autor:** Jonathan Javier Monsalve Giraldo (A01840272)  
**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 24 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025  
**Modalidad:** Individual

## Objetivo

Aplicar un algoritmo de aprendizaje supervisado en PySpark MLlib sobre una muestra M' derivada de la muestra estratificada M construida en la Etapa 2 del proyecto del equipo. El problema elegido es regresión sobre `fare_amount` (tarifa base registrada del viaje), enmarcado como un modelo operativo de auditoría tarifaria a partir de variables del registro del viaje.

## Estructura del notebook

1. **Introducción**: aprendizaje supervisado, algoritmos representativos y los disponibles en PySpark MLlib.
2. **Selección de los datos**: reconstrucción compacta de M (recap de la Etapa 2) y construcción de la muestra individual M'.
3. **Preparación del conjunto de entrenamiento y prueba**: partición estratificada train/test y validación del split.
4. **Construcción de modelos de aprendizaje**: definición del problema, features, pipeline, modelos y resultados.

### Nota para el profesor

La Sección 2.0 reproduce de forma compacta la muestra M de la Etapa 2. El aporte individual inicia en la **Sección 2.1**, con la construcción de M', la partición train/test y el modelado supervisado.

## 1. Introducción

### 1.1 Aprendizaje supervisado

El aprendizaje supervisado es el paradigma en el que un modelo aprende una relación entre un conjunto de variables predictoras y una variable objetivo conocida en los datos de entrenamiento, para después aplicar esa relación a observaciones nuevas. Cada fila del conjunto de entrenamiento incluye tanto las predictoras como la respuesta etiquetada, y la calidad del modelo se evalúa sobre datos no usados durante el ajuste para estimar su capacidad de generalización.

Se distinguen dos grandes tareas según la naturaleza de la variable objetivo. En **regresión** la respuesta es continua, como el monto de una tarifa, la demanda esperada o la temperatura. En **clasificación** la respuesta es categórica, binaria o multiclase, como fraude vs no fraude, especie de una flor o tipo de pago. Esta actividad aborda un problema de regresión.

### 1.2 Algoritmos representativos

La literatura agrupa los algoritmos supervisados en familias con supuestos y compromisos distintos:

- **Modelos lineales** (regresión lineal, regresión logística): coeficientes interpretables, supuesto de linealidad, sensibles a multicolinealidad y a la escala de los predictores. Útiles como baseline y cuando la relación esperada es aproximadamente lineal.
- **Árboles de decisión**: particiones recursivas del espacio de features, capturan no linealidades e interacciones, fáciles de interpretar como reglas; un árbol único tiende a sobreajustar si la profundidad crece.
- **Ensembles de árboles** (Random Forest, Gradient Boosted Trees): combinan muchos árboles para reducir varianza o sesgo. Suelen rendir bien en datos tabulares y exponen importancia de variables, a costa de menor interpretabilidad y mayor costo de cómputo.
- **Máquinas de vectores de soporte (SVM)**: separan clases maximizando el margen, eficaces con datos de alta dimensión y kernels para relaciones no lineales; menos comunes en Big Data por su costo cuadrático.
- **Perceptrón multicapa (MLP)**: redes feedforward capaces de aprender relaciones complejas; requieren más datos, tuning cuidadoso y aportan poca interpretabilidad directa.
- **Naive Bayes**: clasificador probabilístico basado en independencia condicional entre features; rápido y útil en problemas de conteo o texto.

### 1.3 Disponibles en PySpark MLlib

PySpark expone los algoritmos anteriores a través del módulo moderno `pyspark.ml`, basado en DataFrames y organizado bajo el patrón Estimator-Transformer-Pipeline. Un *Estimator* aprende parámetros con `fit()` (por ejemplo, `LinearRegression`, `RandomForestClassifier`); un *Transformer* aplica una transformación con `transform()` (por ejemplo, un modelo ya entrenado o un `VectorAssembler`); un *Pipeline* encadena pasos para garantizar que el mismo preprocesamiento aprendido en train se aplique a test, evitando fuga de información.

| Tarea | Submódulo | Algoritmos disponibles |
|---|---|---|
| Regresión | `pyspark.ml.regression` | `LinearRegression`, `GeneralizedLinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`, `GBTRegressor`, `IsotonicRegression`, `AFTSurvivalRegression`, `FMRegressor` |
| Clasificación | `pyspark.ml.classification` | `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier`, `NaiveBayes`, `LinearSVC`, `OneVsRest`, `FMClassifier` |
| Evaluación | `pyspark.ml.evaluation` | `RegressionEvaluator`, `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator` |
| Tuning | `pyspark.ml.tuning` | `ParamGridBuilder`, `CrossValidator`, `TrainValidationSplit` |

Esta actividad utiliza `LinearRegression` como baseline interpretable y `RandomForestRegressor` como modelo principal, con `RegressionEvaluator` para las métricas y un `Pipeline` para el preprocesamiento.

### 1.4 Referencias

Apache Software Foundation. (2026). *MLlib (DataFrame-based): PySpark 4.1.2 documentation*. https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html

Apache Software Foundation. (2026). *RegressionEvaluator: PySpark 4.1.2 documentation*. https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html

Géron, A. (2022). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow: Concepts, tools, and techniques to build intelligent systems* (3rd ed.). O'Reilly Media.

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media.

## 2. Selección de los datos

### 2.0 Reconstrucción compacta de la muestra M (recap de Etapa 2)

> **Nota al profesor:** las próximas cinco celdas reproducen el muestreo estratificado de la **Etapa 2** del proyecto (downcast del esquema, filtros destructivos, imputaciones con auditoría de nulos, construcción del `stratum_id` y `sampleBy` con piso por estrato). Si está familiarizado con esa entrega, puede saltar directamente a la **Sección 2.1**, donde construyo la muestra individual M' a partir de M con ventana exacta. La 2.0 está aquí para que el notebook sea autocontenido y reproducible en Colab.

El bloque se ejecuta en 5 celdas: (a) setup de Spark y rutas; (b) descarga idempotente de los 24 parquets, downcast del esquema y filtros destructivos; (c) imputaciones de seis columnas y auditoría de nulos; (d) construcción del estrato y derivación de `stratum_id`; (e) recálculo del diccionario de fracciones desde primeros principios y extracción de M vía `sampleBy`.

In [1]:
# (a) setup de Spark y rutas
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pathlib import Path
import urllib.request

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.debug.maxToStringFields", 100)
    .getOrCreate())

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 18:19:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1


In [2]:
# (b) descarga idempotente de los 24 parquets, downcast del esquema y filtros destructivos
CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)

def fetch(url, target):
    if target.exists():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)

for y in YEARS:
    for m in range(1, 13):
        f = f"yellow_tripdata_{y}-{m:02d}.parquet"
        fetch(f"{CDN_BASE}/trip-data/{f}", DATA_DIR / f)
fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")

df_native = (spark.read.option("mergeSchema", "true")
    .parquet(*sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))))
zones = (spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_raw, n_filtered = df_raw.count(), df_filtered.count()
print(f"Crudo: {n_raw:,} | Tras filtros: {n_filtered:,} | Perdida: {(n_raw - n_filtered) / n_raw * 100:.2f}%")
assert (n_raw - n_filtered) / n_raw < 0.15

Crudo: 89,892,322 | Tras filtros: 84,437,138 | Perdida: 6.07%


In [3]:
# (c) imputaciones de seis columnas y auditoría de nulos
df_clean = (df_filtered
    .withColumn("passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte")))
    .withColumn("cbd_congestion_fee",
        F.when(F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
               F.lit(0.0).cast("float"))
         .otherwise(F.col("cbd_congestion_fee")))
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F"))))

imputed_cols = ["passenger_count", "cbd_congestion_fee", "congestion_surcharge",
                "Airport_fee", "RatecodeID", "store_and_fwd_flag"]
nulls = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
assert all((nulls[c] or 0) == 0 for c in imputed_cols), f"Nulos remanentes: {nulls.asDict()}"
print("Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.")

Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.


In [4]:
# (d) construcción del estrato y derivación de `stratum_id`
airport_ids = {r.LocationID for r in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {r.LocationID for r in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {r.LocationID for r in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

df_feat = (df_clean
    .withColumn("pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
         .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
         .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
         .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
         .otherwise("unknown"))
    .withColumn("payment_group",
        F.when(F.col("payment_type") == 0, "flex")
         .when(F.col("payment_type") == 1, "credit")
         .when(F.col("payment_type") == 2, "cash")
         .otherwise("other"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
         .when(F.col("dow").isin(1, 7), "weekend")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
         .otherwise("other"))
    .withColumn("trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
         .when(F.col("trip_distance") < 12.43, "medium")
         .otherwise("long"))
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn("cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd").otherwise("post_cbd"))
    .withColumn("stratum_id",
        F.concat_ws("|",
            F.col("pu_macro_zone"), F.col("payment_group"),
            F.col("day_hour_bucket"), F.col("trip_distance_bin"))))
print("Variables de estrato construidas:", sorted(set(df_feat.columns) - set(df_clean.columns)))

Variables de estrato construidas: ['cbd_period_flag', 'day_hour_bucket', 'dow', 'is_flex_fare', 'payment_group', 'pickup_hour', 'pu_macro_zone', 'stratum_id', 'trip_distance_bin']


In [5]:
# (e) recálculo del diccionario de fracciones desde primeros principios y extracción de M vía `sampleBy`
ESTIMATED_M = 5_030_141
N_M_TARGET = 5_000_000
MIN_FLOOR_M = 500

strata_D = (df_feat.groupBy("stratum_id").count()
    .withColumnRenamed("count", "n_D")
    .withColumn("target_n",
        F.least(F.col("n_D"),
                F.greatest(F.lit(MIN_FLOOR_M).cast("long"),
                           F.round(F.lit(N_M_TARGET) * F.col("n_D") / F.lit(n_filtered)).cast("long"))))
    .withColumn("fraction", F.col("target_n") / F.col("n_D")))

fractions = {r["stratum_id"]: float(r["fraction"]) for r in strata_D.select("stratum_id", "fraction").collect()}
assert all(0 < f <= 1.0 for f in fractions.values())

M = df_feat.stat.sampleBy("stratum_id", fractions, seed=42).cache()
n_M = M.count()
print(f"|M| = {n_M:,} (objetivo {N_M_TARGET:,}, esperado ~5.03M)")
assert abs(n_M - ESTIMATED_M) / ESTIMATED_M < 0.02, f"|M| diverge: {n_M:,}"

# Fin de reconstrucción Etapa 1 y 2
M.select("stratum_id", "fare_amount", "trip_distance", "pu_macro_zone", "payment_group").show(5, truncate=False)

|M| = 5,029,725 (objetivo 5,000,000, esperado ~5.03M)
+--------------------------------------+-----------+-------------+-------------+-------------+
|stratum_id                            |fare_amount|trip_distance|pu_macro_zone|payment_group|
+--------------------------------------+-----------+-------------+-------------+-------------+
|manhattan|credit|late_night|medium    |22.6       |5.72         |manhattan    |credit       |
|manhattan|credit|late_night|medium    |35.9       |7.2          |manhattan    |credit       |
|manhattan|credit|late_night|medium    |16.3       |3.67         |manhattan    |credit       |
|outer_borough|credit|late_night|medium|14.2       |2.67         |outer_borough|credit       |
|manhattan|credit|other|short          |8.6        |0.87         |manhattan    |credit       |
+--------------------------------------+-----------+-------------+-------------+-------------+
only showing top 5 rows


## 2.1 Construcción de M' a partir de M

A partir de M (5.03M filas) construimos M', una submuestra individual de tamaño manejable que preserva el diseño estratificado de Etapa 2. La técnica es **ventana exacta por estrato**:

- Para cada uno de los 240 estratos s (valores únicos de `stratum_id`), calculamos `target_n_s = max(50, floor(0.20 * n_M_s))`. El piso de 50 garantiza determinísticamente que un split 80/20 posterior (Sección 3) deje al menos 10 filas en test, incluso en los estratos más chicos heredados del piso de Etapa 2.
- Sobre `Window.partitionBy("stratum_id").orderBy(F.rand(seed=42))` asignamos `row_number()` y filtramos `rn <= target_n_s`. Esto materializa exactamente `target_n_s` filas por estrato.

El método es simétrico al split train/test de la Sección 3 (mismo patrón de ventana) y consistente con la inversión estratificada de Etapa 2.

In [6]:
# construcción de M' por ventana exacta con piso determinístico de 50
F_GLOBAL_MP = 0.20
MIN_FLOOR_MP = 50

counts_M = M.groupBy("stratum_id").count().withColumnRenamed("count", "n_M_s")
target_Mp = counts_M.withColumn(
    "target_n_s",
    F.least(
        F.col("n_M_s"),
        F.greatest(F.lit(MIN_FLOOR_MP).cast("long"),
                   F.floor(F.lit(F_GLOBAL_MP) * F.col("n_M_s")).cast("long"))))

w_Mp = Window.partitionBy("stratum_id").orderBy(F.rand(seed=42))
M_prime = (M.join(target_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_Mp))
    .filter(F.col("rn") <= F.col("target_n_s"))
    .drop("rn", "target_n_s", "n_M_s")
    .cache())

n_Mp = M_prime.count()
print(f"|M'| = {n_Mp:,} (esperado ~1.0M)")
assert 950_000 <= n_Mp <= 1_100_000, f"|M'| fuera de rango: {n_Mp:,}"

|M'| = 1,006,344 (esperado ~1.0M)


## 2.2 Validación de representatividad M' vs M

Tres validaciones obligatorias para verificar que M' preserva la estructura estratificada de M:

1. **Tamaños y piso**: |M|, |M'|, y conteo mínimo por estrato en M' (debe ser >= 50 por el piso determinístico).
2. **Cardinalidad de estratos**: M' debe contener los 240 estratos de M.
3. **Marginales** en las cuatro variables del estrato (`pu_macro_zone`, `payment_group`, `day_hour_bucket`, `trip_distance_bin`): desviación máxima entre p_M y p_M' por categoría.

In [7]:
# validación compacta M' vs M
n_M_v, n_Mp_v = M.count(), M_prime.count()
strata_M = M.select("stratum_id").distinct().count()
strata_Mp = M_prime.select("stratum_id").distinct().count()
min_count_Mp = M_prime.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"|M| = {n_M_v:,}  |M'| = {n_Mp_v:,}  ratio = {n_Mp_v / n_M_v:.4f}")
print(f"Estratos en M = {strata_M}, en M' = {strata_Mp} (debe ser 240)")
print(f"Piso mínimo por estrato en M' = {min_count_Mp} (debe ser >= 50)\n")
assert strata_Mp == strata_M, "M' perdió estratos"
assert min_count_Mp >= 50, f"Piso violado: {min_count_Mp}"

# Marginales en las 4 variables del estrato
for col in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_M = {r[col]: r["count"] / n_M_v for r in M.groupBy(col).count().collect()}
    p_Mp = {r[col]: r["count"] / n_Mp_v for r in M_prime.groupBy(col).count().collect()}
    max_diff_pp = max(abs(p_M.get(k, 0) - p_Mp.get(k, 0)) for k in set(p_M) | set(p_Mp)) * 100
    print(f"  {col}: max |p_M - p_M'| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col} diverge: {max_diff_pp:.4f} pp"

|M| = 5,029,725  |M'| = 1,006,344  ratio = 0.2001
Estratos en M = 240, en M' = 240 (debe ser 240)
Piso mínimo por estrato en M' = 50 (debe ser >= 50)

  pu_macro_zone: max |p_M - p_M'| = 0.0369 pp
  payment_group: max |p_M - p_M'| = 0.0307 pp
  day_hour_bucket: max |p_M - p_M'| = 0.0079 pp
  trip_distance_bin: max |p_M - p_M'| = 0.0287 pp


## 3. Preparación del conjunto de entrenamiento y prueba

**Justificación del 80/20**: la proporción se elige por su interacción exacta con el piso de 50 filas por estrato heredado de la sección 2.1. `floor(0.8 * 50) = 40` filas de train y `50 - 40 = 10` filas de test en los estratos más chicos: train suficiente para que el modelo vea el caso raro y test con el mínimo razonable para una estadística por estrato. Un split más agresivo (90/10) dejaría 5 filas en test, insuficiente; uno más conservador (70/30) restaría 5 filas de train sin ganancia útil dado que 10 ya es el piso adecuado. Sobre el total (M' ~1M filas), el conjunto de test de ~200k tiene precisión más que suficiente para estimar RMSE, MAE y R² con error estándar despreciable. El 80/20 también es la convención estándar para conjuntos medianos a grandes en aprendizaje supervisado.

**Técnica**: ventana exacta por estrato, mismo patrón usado en 2.1 para construir M': `Window.partitionBy("stratum_id").orderBy(F.rand(seed=123))` con `row_number()` y filtro `rn <= floor(0.8 * n_s)` para train, complemento para test. Esto materializa exactamente `floor(0.8 * n_Mp_s)` filas de train y `n_Mp_s - floor(0.8 * n_Mp_s)` de test por estrato. El seed difiere del usado en 2.1 (42) para mantener independencia metodológica entre las dos etapas de muestreo.

**¿Por qué no `randomSplit([0.8, 0.2])`?**: Bernoulli puro por fila, ignora la columna de estrato; deja varianza muestral que en estratos chicos (53 filas en el extremo de Etapa 2) puede romper la inversión estratificada.

**¿Por qué no `sampleBy("stratum_id", {sid: 0.8})`?**: también Bernoulli, ahora por estrato; no garantiza conteo exacto. La ventana exacta es el equivalente PySpark del `stratify=y` de scikit-learn.

In [8]:
# split estratificado exacto 80/20 sobre M_prime
TRAIN_RATIO = 0.8

counts_Mp = M_prime.groupBy("stratum_id").count().withColumnRenamed("count", "n_Mp_s")
w_split = Window.partitionBy("stratum_id").orderBy(F.rand(seed=123))

Mp_with_rn = (M_prime.join(counts_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_split))
    .withColumn("train_cutoff", F.floor(F.lit(TRAIN_RATIO) * F.col("n_Mp_s")).cast("long")))

train_df = (Mp_with_rn.filter(F.col("rn") <= F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())
test_df  = (Mp_with_rn.filter(F.col("rn") >  F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())

n_train, n_test = train_df.count(), test_df.count()
print(f"|train| = {n_train:,}  |test| = {n_test:,}  ratio_train = {n_train / (n_train + n_test):.4f}")

|train| = 804,991  |test| = 201,353  ratio_train = 0.7999


In [9]:
# verificación post-split: cardinalidad, piso y marginales train vs test
strata_train = train_df.select("stratum_id").distinct().count()
strata_test = test_df.select("stratum_id").distinct().count()
min_train = train_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]
min_test = test_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"Estratos train = {strata_train}, test = {strata_test} (esperado 240)")
print(f"Piso min en train = {min_train}, en test = {min_test} (test debe ser >= 10)\n")
assert strata_train == 240 and strata_test == 240
assert min_test >= 10, f"Piso en test violado: {min_test}"

# Marginales train vs test en las 4 variables del estrato
for col in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_tr = {r[col]: r["count"] / n_train for r in train_df.groupBy(col).count().collect()}
    p_te = {r[col]: r["count"] / n_test for r in test_df.groupBy(col).count().collect()}
    max_diff_pp = max(abs(p_tr.get(k, 0) - p_te.get(k, 0)) for k in set(p_tr) | set(p_te)) * 100
    print(f"  {col}: max |p_train - p_test| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col} diverge: {max_diff_pp:.4f} pp"

Estratos train = 240, test = 240 (esperado 240)
Piso min en train = 40, en test = 10 (test debe ser >= 10)

  pu_macro_zone: max |p_train - p_test| = 0.0315 pp
  payment_group: max |p_train - p_test| = 0.0215 pp
  day_hour_bucket: max |p_train - p_test| = 0.0066 pp
  trip_distance_bin: max |p_train - p_test| = 0.0176 pp


## 4. Construcción de modelos de aprendizaje

### 4.1 Definición del problema y variable objetivo

El problema supervisado es **regresión sobre `fare_amount`**, la tarifa base registrada del viaje. La motivación es un modelo operativo de auditoría tarifaria: dada la combinación de zona de origen, código de tarifa, régimen Flex, horario, fecha y distancia recorrida, ¿qué tarifa base es coherente con esos atributos?

Para evitar fuga de información se excluyen **nueve columnas**: `total_amount` (suma que contiene a `fare_amount`) y sus ocho componentes aritméticos (`tip_amount`, `tolls_amount`, `extra`, `mta_tax`, `improvement_surcharge`, `congestion_surcharge`, `Airport_fee`, `cbd_congestion_fee`). `total_amount` contiene directamente a `fare_amount`; los demás componentes son variables posteriores al cobro y se excluyen para evitar fuga de información o contexto de facturación no disponible antes de evaluar la tarifa base.

### 4.2 Selección de features

Siete predictores, todos disponibles en el registro del viaje:

| Feature | Tipo | Justificación |
|---|---|---|
| `trip_distance` | numérica continua | Predictor dominante; correlación 0.93 con `fare_amount` medida en Etapa 2 sección 6.2 |
| `pu_macro_zone` | categórica (4) | Captura tarifa plana aeroportuaria y diferenciación geográfica de origen |
| `RatecodeID` | categórica (7) | Selector del régimen tarifario (JFK flat, Newark, Negotiated, Flex) |
| `is_flex_fare` | binaria | Marca régimen upfront pricing sin taxímetro |
| `day_hour_bucket` | categórica (5) | Captura patrones temporales de operación y posibles cambios de régimen tarifario |
| `cbd_period_flag` | binaria | Discontinuidad regulatoria del CBD fee desde 2025-01-05 |
| `passenger_count` | numérica entera | Bajo costo; ocasionalmente cambia el régimen de tarifa (viajes grupales) |

In [10]:
# definición de target, features y verificación de ausencia de fuga
TARGET = "fare_amount"
FEATURE_COLS = [
    "trip_distance", "pu_macro_zone", "RatecodeID", "is_flex_fare",
    "day_hour_bucket", "cbd_period_flag", "passenger_count",
]
LEAKAGE_COLS = {"total_amount", "tip_amount", "tolls_amount", "extra", "mta_tax",
                "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"}

assert not (set(FEATURE_COLS) & LEAKAGE_COLS), "Hay fuga en FEATURE_COLS"
assert TARGET not in FEATURE_COLS, "El target no debe estar entre las features"

print(f"Target: {TARGET}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Columnas excluidas por fuga ({len(LEAKAGE_COLS)}): {sorted(LEAKAGE_COLS)}")

Target: fare_amount
Features (7): ['trip_distance', 'pu_macro_zone', 'RatecodeID', 'is_flex_fare', 'day_hour_bucket', 'cbd_period_flag', 'passenger_count']
Columnas excluidas por fuga (9): ['Airport_fee', 'cbd_congestion_fee', 'congestion_surcharge', 'extra', 'improvement_surcharge', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']


### 4.3 Pipeline de preprocesamiento

El pipeline de `pyspark.ml` encadena tres transformadores antes de cada modelo:

1. **StringIndexer** sobre las cuatro categóricas (`pu_macro_zone`, `RatecodeID`, `day_hour_bucket`, `cbd_period_flag`): asigna un índice numérico a cada categoría. Es estimador (aprende el mapeo en train, lo aplica en test).
2. **OneHotEncoder** sobre los índices: expande a vectores binarios para evitar que el modelo lineal interprete los índices como valores ordinales. Para árboles no es estrictamente necesario, pero mantenerlo en el mismo pipeline permite comparación directa entre LR y RF sobre la misma representación.
3. **VectorAssembler**: concatena las numéricas (`trip_distance`, `is_flex_fare`, `passenger_count`) y las OHE en un único vector `features`.

Usar un solo `Pipeline` garantiza que el ajuste se haga **solo sobre train**. Aplicar `transform` sobre test usa los índices y la codificación aprendidos en train; cualquier categoría nueva en test cae bajo `handleInvalid="keep"`.

In [11]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Cast booleano a byte para compatibilidad con VectorAssembler
train_ml = train_df.withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))
test_ml  = test_df.withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))

CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket", "cbd_period_flag"]
NUM_COLS = ["trip_distance", "is_flex_fare", "passenger_count"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CAT_COLS]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in CAT_COLS],
                       outputCols=[f"{c}_ohe" for c in CAT_COLS])
assembler = VectorAssembler(inputCols=NUM_COLS + [f"{c}_ohe" for c in CAT_COLS],
                           outputCol="features", handleInvalid="keep")

preprocessing_stages = indexers + [encoder, assembler]
print(f"Etapas de preprocesamiento: {len(preprocessing_stages)} ({len(CAT_COLS)} StringIndexers + 1 OneHotEncoder + 1 VectorAssembler)")

Etapas de preprocesamiento: 6 (4 StringIndexers + 1 OneHotEncoder + 1 VectorAssembler)


### 4.4 Modelo baseline: LinearRegression

`LinearRegression` ajusta una combinación lineal de las features (intercepto + suma ponderada) minimizando el error cuadrático medio. Supuestos: la relación entre features y target es razonablemente lineal en cada régimen tarifario (cuestionable globalmente por la discontinuidad del JFK flat fare, pero defendible como baseline). El modelo es interpretable: cada coeficiente da el efecto estimado de un cambio unitario en su feature, manteniendo las demás constantes.

Hiperparámetros: `regParam=0.0` y `elasticNetParam=0.0` (OLS sin regularización), `maxIter=100`. Se usa `regParam=0` como baseline OLS no regularizado; el warning de matriz singular que Spark emite se reporta como limitación del baseline, probablemente asociado a colinealidad o casi colinealidad en la representación categórica. No se optimiza LR porque su función aquí es servir como referencia interpretable frente al bosque aleatorio.

In [12]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import time

lr = LinearRegression(featuresCol="features", labelCol="fare_amount",
                      regParam=0.0, elasticNetParam=0.0, maxIter=100)
pipeline_lr = Pipeline(stages=preprocessing_stages + [lr])

t0 = time.time()
model_lr = pipeline_lr.fit(train_ml)
print(f"LR fit time: {time.time() - t0:.1f}s")

pred_lr = model_lr.transform(test_ml)
ev_rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse")
ev_mae  = RegressionEvaluator(labelCol="fare_amount", metricName="mae")
ev_r2   = RegressionEvaluator(labelCol="fare_amount", metricName="r2")
rmse_lr, mae_lr, r2_lr = ev_rmse.evaluate(pred_lr), ev_mae.evaluate(pred_lr), ev_r2.evaluate(pred_lr)
print(f"LR test  RMSE={rmse_lr:.3f}  MAE={mae_lr:.3f}  R2={r2_lr:.4f}")

# Coeficientes con sus nombres
lr_m = model_lr.stages[-1]
feature_names = list(NUM_COLS)
for cat, ix in zip(CAT_COLS, model_lr.stages[:len(CAT_COLS)]):
    feature_names.extend(f"{cat}={lbl}" for lbl in ix.labels)

coefs = lr_m.coefficients.toArray()
print(f"\nIntercept: {lr_m.intercept:.3f} USD")
print(f"Coeficientes ({len(coefs)} features):")
for name, c in zip(feature_names, coefs):
    print(f"  {name:30s} {c:+.4f}")

26/05/24 18:20:36 WARN Instrumentation: [efd5e7a9] regParam is zero, which might cause numerical instability and overfitting.
26/05/24 18:20:36 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/24 18:20:37 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/05/24 18:20:37 WARN Instrumentation: [efd5e7a9] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


LR fit time: 3.8s
LR test  RMSE=6.558  MAE=3.615  R2=0.8645

Intercept: 7.934 USD
Coeficientes (20 features):
  trip_distance                  +3.3976
  is_flex_fare                   +3.5026
  passenger_count                +0.2156
  pu_macro_zone=manhattan        -0.3396
  pu_macro_zone=airport          +0.9012
  pu_macro_zone=outer_borough    -0.5177
  pu_macro_zone=unknown          +0.0065
  RatecodeID=1                   -0.4126
  RatecodeID=99                  -2.4267
  RatecodeID=2                   +0.6570
  RatecodeID=5                   +38.4559
  RatecodeID=3                   +22.5934
  RatecodeID=4                   +34.6235
  day_hour_bucket=other          +0.5165
  day_hour_bucket=weekend        -0.2643
  day_hour_bucket=weekday_pm_peak +0.4393
  day_hour_bucket=weekday_am     +0.0984
  day_hour_bucket=late_night     -1.9510
  cbd_period_flag=post_cbd       -0.0704
  cbd_period_flag=pre_cbd        +0.0704


### 4.5 Modelo principal: RandomForestRegressor con tuning

Random Forest entrena un ensemble de árboles de regresión sobre submuestras bootstrap de train y subconjuntos aleatorios de features; la predicción es el promedio de las predicciones individuales. Captura no linealidades e interacciones implícitamente sin requerir feature engineering manual.

Supuestos relajados vs LinearRegression: no asume linealidad ni homoscedasticidad, no es sensible a la escala de features, robusto a outliers moderados. La interpretabilidad se limita a `featureImportances` (reducción promedio de impurity por feature).

Para RF sí aplicamos tuning de hiperparámetros con `TrainValidationSplit` y una grilla compacta:

- `numTrees in [30, 50]`: balancea estabilidad y costo
- `maxDepth in [6, 8]`: brackets del rango usual de profundidad para datos tabulares
- `subsamplingRate in [0.8, 1.0]`: el default de Spark es 1.0 (cada árbol entrena con el 100% del train); bajarlo a 0.8 introduce más decorrelación entre árboles, una técnica clásica de Random Forest que sklearn históricamente usaba por default

Total: 8 combinaciones con `trainRatio=0.75`. El mejor modelo se selecciona por RMSE en la validación interna (25% de train) y se evalúa una sola vez en test al final. Se omite `CrossValidator` porque con 800k filas la varianza entre folds es despreciable y el costo (3x el de TVS) no se justifica.

In [13]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

rf = RandomForestRegressor(featuresCol="features", labelCol="fare_amount",
                           featureSubsetStrategy="auto", seed=42)
pipeline_rf = Pipeline(stages=preprocessing_stages + [rf])

param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 50])
    .addGrid(rf.maxDepth, [6, 8])
    .addGrid(rf.subsamplingRate, [0.8, 1.0])
    .build())

tvs = TrainValidationSplit(estimator=pipeline_rf, estimatorParamMaps=param_grid,
                           evaluator=ev_rmse, trainRatio=0.75, parallelism=2, seed=42)

t0 = time.time()
tvs_model = tvs.fit(train_ml)
print(f"TVS fit time (8 combinaciones): {time.time() - t0:.1f}s")

# Grilla con RMSE de validación interna
print("\nGrid de hiperparámetros (RMSE en validación interna 25%):")
val_rmses = tvs_model.validationMetrics
for params, rmse in zip(param_grid, val_rmses):
    combo = {p.name: v for p, v in params.items()}
    print(f"  {combo} -> RMSE val: {rmse:.4f}")

best_idx = min(range(len(val_rmses)), key=lambda i: val_rmses[i])
best_combo = {p.name: v for p, v in param_grid[best_idx].items()}
print(f"\nMejor configuración: {best_combo}")

# Métricas test con el bestModel (TVS ya refit en train completo)
best_pipeline_rf = tvs_model.bestModel
pred_rf = best_pipeline_rf.transform(test_ml)
rmse_rf, mae_rf, r2_rf = ev_rmse.evaluate(pred_rf), ev_mae.evaluate(pred_rf), ev_r2.evaluate(pred_rf)
print(f"RF (best) test  RMSE={rmse_rf:.3f}  MAE={mae_rf:.3f}  R2={r2_rf:.4f}")

# Importancia de features (todas, en orden de feature)
rf_m = best_pipeline_rf.stages[-1]
fi = rf_m.featureImportances.toArray()
print(f"\nfeatureImportances ({len(fi)} features):")
for name, imp in zip(feature_names, fi):
    print(f"  {name:30s} {imp:.4f}")

26/05/24 18:21:14 WARN DAGScheduler: Broadcasting large task binary with size 1258.5 KiB
26/05/24 18:21:16 WARN DAGScheduler: Broadcasting large task binary with size 1243.8 KiB
26/05/24 18:21:55 WARN DAGScheduler: Broadcasting large task binary with size 1168.6 KiB
26/05/24 18:21:58 WARN DAGScheduler: Broadcasting large task binary with size 1959.4 KiB
26/05/24 18:22:04 WARN DAGScheduler: Broadcasting large task binary with size 1163.4 KiB
26/05/24 18:22:06 WARN DAGScheduler: Broadcasting large task binary with size 1939.9 KiB
26/05/24 18:22:21 WARN DAGScheduler: Broadcasting large task binary with size 1257.8 KiB


TVS fit time (8 combinaciones): 103.9s

Grid de hiperparámetros (RMSE en validación interna 25%):
  {'numTrees': 30, 'maxDepth': 6, 'subsamplingRate': 0.8} -> RMSE val: 7.0137
  {'numTrees': 30, 'maxDepth': 6, 'subsamplingRate': 1.0} -> RMSE val: 7.0341
  {'numTrees': 30, 'maxDepth': 8, 'subsamplingRate': 0.8} -> RMSE val: 6.5845
  {'numTrees': 30, 'maxDepth': 8, 'subsamplingRate': 1.0} -> RMSE val: 6.6494
  {'numTrees': 50, 'maxDepth': 6, 'subsamplingRate': 0.8} -> RMSE val: 7.0664
  {'numTrees': 50, 'maxDepth': 6, 'subsamplingRate': 1.0} -> RMSE val: 7.0246
  {'numTrees': 50, 'maxDepth': 8, 'subsamplingRate': 0.8} -> RMSE val: 6.6160
  {'numTrees': 50, 'maxDepth': 8, 'subsamplingRate': 1.0} -> RMSE val: 6.6082

Mejor configuración: {'numTrees': 30, 'maxDepth': 8, 'subsamplingRate': 0.8}
RF (best) test  RMSE=6.399  MAE=3.352  R2=0.8710

featureImportances (20 features):
  trip_distance                  0.5620
  is_flex_fare                   0.0061
  passenger_count                0.0

### 4.6 Comparación y análisis de residuales

Tabla comparativa de los dos modelos en el conjunto de test. La elección entre LR y RF se basa principalmente en RMSE (penaliza errores grandes, importante para la cola de tarifas aeroportuarias) y R² (fracción de varianza explicada).

Análisis de residuales por `is_flex_fare`: el régimen Flex tiene pricing upfront declarado al inicio del viaje, no calculado por el taxímetro. Es esperable que ambos modelos predigan peor en Flex porque la tarifa final no sigue la mecánica taxímetro-distancia-tiempo de los viajes en régimen Metered. Si los residuales son sistemáticamente mayores en `is_flex_fare = True`, eso justifica filtrar Flex o entrenar modelos separados en una versión futura.

Lectura de negocio: para un modelo operativo de auditoría tarifaria, un MAE < 1 USD es excelente, 1-3 USD aceptable, > 3 USD problemático. RMSE puede ser mayor por la cola de tarifas aeroportuarias.

In [14]:
import pandas as pd

# Tabla comparativa
print("Comparación en test:")
print(pd.DataFrame({
    "Modelo": ["LinearRegression", "RandomForestRegressor (tuned)"],
    "RMSE": [rmse_lr, rmse_rf], "MAE": [mae_lr, mae_rf], "R2": [r2_lr, r2_rf],
}).to_string(index=False, float_format="%.4f"))

# Métricas RF por régimen tarifario (validación empírica del 4.6)
print("\nMétricas RF por régimen tarifario:")
for label, val in [("Metered", 0), ("Flex", 1)]:
    pred_g = pred_rf.filter(F.col("is_flex_fare") == val)
    mae_g  = ev_mae.evaluate(pred_g)
    rmse_g = ev_rmse.evaluate(pred_g)
    print(f"  {label}: MAE={mae_g:.3f}  RMSE={rmse_g:.3f}")

Comparación en test:
                       Modelo   RMSE    MAE     R2
             LinearRegression 6.5585 3.6148 0.8645
RandomForestRegressor (tuned) 6.3994 3.3519 0.8710

Métricas RF por régimen tarifario:
  Metered: MAE=2.959  RMSE=5.861
  Flex: MAE=5.590  RMSE=8.862


## Cierre

**¿El modelo funciona? Sí, condicionado al régimen tarifario.** Sobre viajes en régimen Metered, el MAE de 2.96 USD equivale a un error del 15-20% sobre tarifas típicas de Manhattan, por lo que el modelo puede servir como primer filtro de auditoría tarifaria. Sobre viajes Flex Fare el MAE se duplica a 5.59 USD; ahí el modelo unificado pierde utilidad práctica y la recomendación es entrenar separado por régimen. R² agregada de 0.87 indica que las 7 features pre-medidor capturan buena parte de la estructura tarifaria sin acceso a los componentes de cobro.

**Resultados clave**

El modelo final es un bosque aleatorio (`numTrees=30, maxDepth=8, subsamplingRate=0.8`), seleccionado por `TrainValidationSplit` sobre 8 combinaciones. En el conjunto de prueba: RMSE 6.40 USD, MAE 3.35 USD, R² 0.87. La regresión lineal baseline: RMSE 6.56 USD, MAE 3.61 USD, R² 0.86.

El RMSE (6.40 USD) casi duplica al MAE porque la métrica cuadrática penaliza fuertemente los errores grandes: la diferencia entre RMSE y MAE sugiere una cola de viajes específicos (flat rates aeroportuarios, tarifas negociadas) donde el error puede ser bastante mayor al promedio absoluto.

Tres lecturas accionables:

1. **El bosque aleatorio aporta poco sobre la regresión lineal en agregado.** La ganancia (0.26 USD en MAE, 0.16 USD en RMSE) probablemente viene de capturar discontinuidades específicas (tarifas planas JFK y Newark) que la regresión lineal aproxima con coeficientes promedio. En este conjunto, una relación lineal con la distancia y ajustes por régimen ya explica gran parte de la varianza. Si el costo de mantener un modelo no lineal en producción es alto, el lineal es defendible como baseline operativo.

2. **El modelo unificado no es recomendable para auditar viajes Flex Fare.** El error casi se duplica en este régimen (MAE 5.59 USD en Flex contra 2.96 USD en Metered) porque la tarifa upfront no sigue la mecánica taxímetro-distancia. En términos operativos, el umbral de alerta tendría que ser más amplio en Flex, reduciendo sensibilidad ante anomalías. Un sistema operativo debería entrenar modelos separados por régimen.

3. **Distancia es la variable más influyente; el resto refina.** El bosque atribuye 56% de su importancia a `trip_distance`, 21% combinado a `pu_macro_zone` (aeropuerto y Manhattan dominan), 17% a `RatecodeID` (tarifas flat). Las variables temporales (`day_hour_bucket`, `cbd_period_flag`) aportan menos de 1% combinado porque son proxies indirectos; la duración real del viaje y las condiciones de tráfico no están incluidas como features.

Del tuning, sólo `maxDepth` mueve la métrica de forma material (6 → 8 baja el RMSE de validación de 7.0 a 6.6). `numTrees` y `subsamplingRate` aportan ganancias marginales en este conjunto.

**Limitaciones**

- El régimen Flex requiere modelo separado; ya cuantificado arriba.
- No se modela el tráfico de forma directa (sólo se aproxima vía `day_hour_bucket`, que el bosque casi ignora); esto explica varianza residual que ningún feature disponible captura.
- La sensibilidad a la semilla aleatoria no se exploró.

**Recomendaciones**

- Validación temporal con datos de 2026 (reservados en Etapa 2 para precisamente esto).
- Modelos separados Flex vs Metered en producción.

**Declaración de uso de IA**

Google. (2026). *Gemini 3.5 Flash* [Modelo de lenguaje grande], utilizado para el proceso de aprendizaje del contenido de la semana y la validación de errores conceptuales y de código. https://deepmind.google/models/gemini/flash/